# Gemma 3n E4B smoke test on free Colab (debug only)

Runtime → Change runtime type → **T4 GPU** before running.

This uses Ollama's quantized `gemma3n:e4b`, so the data are **for debugging only** (they go to `data/gemma3n-debug/`). Formal runs need BF16 on an Ampere-or-newer GPU.

Author: Zhang Qi

In [ ]:
# 1. Install and start Ollama, then download the model (~10 GB, a few minutes)
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull gemma3n:e4b

In [ ]:
# 2. Get the code: upload gemma3n-rationality.zip (works for a private repository)
from google.colab import files
files.upload()
!unzip -q -o gemma3n-rationality.zip
%cd gemma3n-rationality
!pip install -q -r requirements.txt

3. Settings for this test. No API key is needed: Ollama runs inside this notebook.

In [ ]:
%%writefile .env
LLM_PROVIDER=openai
LLM_BASE_URL=http://localhost:11434/v1
LLM_API_KEY=ollama
LLM_MODEL=gemma3n:e4b
LLM_LABEL=gemma-3n-e4b-it (debug)
LLM_RUN_TAG=gemma3n-debug
LLM_ROLE_MODE=system_as_user
LLM_PROMPTS=paper

In [ ]:
# 4. Check the settings were read: expect gemma3n:e4b | gemma3n-debug | .../data/gemma3n-debug/raw | system_as_user
!python -c "import config as C; print(C.MODEL, '|', C.RUN_TAG, '|', C.RAW_DIR, '|', C.ROLE_MODE)"
!python scripts/00_selftest.py | tail -1

In [ ]:
# 5. Smoke test: one subject per domain, full transcript saved to smoke_gemma3n.txt
!python scripts/01_smoke_test.py --condition baseline 2>&1 | tee smoke_gemma3n.txt | tail -40

In [ ]:
# 6. Download the full output and send it
from google.colab import files
files.download('smoke_gemma3n.txt')